# EO/IR Füzyon Projesi: Colab'da YOLOv8n eğitimi

Bu defteri **iki kez** çalıştıracaksın: önce `MODALITY = "ir"`, sonra `"eo"`. Her şey aynı, sadece veri değişiyor. Bu bir kontrollü deney: iki sonuç arasındaki fark modaliteden gelsin, ayardan değil.

**Sıra:** GPU kontrolü, kurulum, Drive, veriyi yerel diske aç, doğrula, YAML, eğitim, eğrileri oku.

**Önce:** Çalışma zamanı (Runtime) > Çalışma zamanı türünü değiştir (Change runtime type) > T4 GPU.


## 1. GPU var mı?
CPU'da bir epoch 44 dakika sürmüştü. GPU yoksa aşağıdaki hücre `False` yazar, devam etme.


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


## 2. Kurulum ve sürüm sabitleme
Yerelde `ultralytics 8.4.131` kullanıyorduk. Aynı sürümü kuruyoruz ki varsayılan ayarlar değişmesin (tekrarlanabilirlik).


In [ ]:
!pip -q install ultralytics==8.4.131
import ultralytics
print(ultralytics.__version__)


## 3. Ayarlar ve Drive
`MODALITY` değiştireceğin tek satır. Drive'da `eo_ir_fusion` adlı bir klasör olmalı, içinde `dataset_ir.zip` ve `dataset_eo.zip`.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

MODALITY = "ir"        # ilk kosu "ir", ikinci kosu "eo"
EPOCHS = 50
RESUME = False         # oturum koptuysa True yap (asagida anlatiliyor)

DRIVE_DIR = "/content/drive/MyDrive/eo_ir_fusion"
ZIP = f"{DRIVE_DIR}/dataset_{MODALITY}.zip"
PROJECT = f"{DRIVE_DIR}/runs"
RUN_NAME = f"{MODALITY}_yolov8n_e{EPOCHS}"

import os
assert os.path.exists(ZIP), f"Zip bulunamadi: {ZIP}"
print("Zip bulundu:", ZIP)


## 4. Veriyi yerel diske aç
Eğitim her epoch'ta on binlerce görüntüyü okur. Drive ağ üzerinden bağlı bir disk ve her dosya okuması yavaş. Zip'i bir kez kopyalayıp Colab'ın yerel diskine (`/content`) açmak eğitimi ciddi hızlandırır.

Bedeli: oturum kapanınca `/content` silinir. Bu yüzden zip Drive'da kalıyor ve ağırlıkları da Drive'a yazacağız.


In [ ]:
!cp "{ZIP}" /content/
!unzip -q -o "/content/dataset_{MODALITY}.zip" -d /content
!ls /content/dataset_{MODALITY}


## 5. Doğrula
Yerelde `prepare_llvip_yolo.py` çıktısında gördüğümüz sayılar: train 9921, val 2104, test 3463. Ayrıca her görüntünün etiketi var mı ve kliplerin kümeler arasında karışmadığı. Bir şey tutmazsa hücre burada durur, eğitime geçme.


In [ ]:
from pathlib import Path

DATA = Path(f"/content/dataset_{MODALITY}")
EXPECTED = {"train": 9921, "val": 2104, "test": 3463}

clips = {}
for split, n_expected in EXPECTED.items():
    imgs = {p.stem for p in (DATA / "images" / split).glob("*.jpg")}
    lbls = {p.stem for p in (DATA / "labels" / split).glob("*.txt")}
    print(f"{split:5s} goruntu={len(imgs):5d} etiket={len(lbls):5d} beklenen={n_expected}")
    assert len(imgs) == n_expected, f"{split}: goruntu sayisi beklenenden farkli"
    assert imgs == lbls, f"{split}: goruntu ve etiket isimleri ayni degil"
    clips[split] = {s[:2] for s in imgs}

assert not (clips["train"] & clips["val"]), "train ve val klip paylasiyor"
assert not (clips["train"] & clips["test"]), "train ve test klip paylasiyor"
assert not (clips["val"] & clips["test"]), "val ve test klip paylasiyor"
print("Tamam: sayilar tutuyor, klipler ayrik.")
print("Val klipleri:", sorted(clips["val"]))


## 6. YAML
Yerel makinedeki YAML'ın yolu Colab'da geçerli değil, o yüzden YAML'ı burada üretiyoruz.


In [ ]:
import yaml

cfg = {
    "path": str(DATA),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "person"},
}
yaml_path = f"/content/dataset_{MODALITY}.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print(open(yaml_path).read())


## 7. Eğitim
Bilinçli seçimler:
- **yolov8n.pt**: COCO'da eğitilmiş 3 milyon parametreli model, transfer learning başlangıcı.
- **IR ve EO için tamamen aynı ayarlar.** Sadece veri farklı.
- **optimizer=AdamW, lr0=0.002 açıkça yazıldı.** Ultralytics'te `optimizer=auto`, toplam iterasyon sayısına bakıp SGD ya da AdamW seçer. Epoch sayısını değiştirdiğinde sessizce başka bir optimizer'a geçebilir. Yerelde auto'nun seçtiği değeri sabitledik.
- **workers=2:** Colab'ın az CPU çekirdeği var.
- **project Drive'da:** ağırlıklar her epoch Drive'a yazılır, oturum kopsa da `last.pt` kalır.
- **patience=20:** val skoru 20 epoch iyileşmezse eğitim durur.

**Oturum koptuysa:** 2-6 numaralı hücreleri yeniden çalıştır, ayarlar hücresinde `RESUME = True` yap, sonra bu hücreyi çalıştır. Eğitim kaldığı epoch'tan devam eder.


In [ ]:
from ultralytics import YOLO

last = Path(PROJECT) / RUN_NAME / "weights" / "last.pt"

if RESUME and last.exists():
    YOLO(str(last)).train(resume=True)
else:
    model = YOLO("yolov8n.pt")
    model.train(
        data=yaml_path,
        epochs=EPOCHS,
        imgsz=640,
        batch=16,
        workers=2,
        optimizer="AdamW",
        lr0=0.002,
        patience=20,
        seed=0,
        project=PROJECT,
        name=RUN_NAME,
    )


## 8. Eğrileri oku
`results.png` içinde şunlara bak:
1. Train ve val `box_loss` / `cls_loss` birlikte düşüyor mu?
2. Val mAP bir yerde düzleşiyor mu? Düzleştiği epoch, kaç epoch'a ihtiyaç olduğunu gösterir.
3. Train loss düşmeye devam ederken val loss yükseliyorsa model ezberlemeye başlamıştır (overfitting).

Val artık klip bazlı olduğu için bu eğriler sahneler arası genellemeyi gösteriyor.


In [ ]:
from IPython.display import Image, display

display(Image(filename=f"{PROJECT}/{RUN_NAME}/results.png"))


## Val metriği
`best.pt`, val skoruna göre seçilen epoch'un ağırlıkları.


In [ ]:
best = f"{PROJECT}/{RUN_NAME}/weights/best.pt"
metrics = YOLO(best).val(data=yaml_path, split="val", imgsz=640, batch=16, workers=2)
print("mAP50:", round(metrics.box.map50, 4), "| mAP50-95:", round(metrics.box.map, 4))


## 9. Test (şimdi çalıştırma)
Test setine bir kez, tüm kararlar bittikten sonra bakacağız: epoch sayısı, ağırlıklar, füzyon kuralı hepsi val ile seçilir. Test'e bakıp bir şeyi değiştirirsen test de val'e dönüşür. Bu yüzden hücre yorum satırı halinde.


In [ ]:
# Hazir oldugunda yorumlari kaldir, TEK SEFER calistir:
# best = f"{PROJECT}/{RUN_NAME}/weights/best.pt"
# YOLO(best).val(data=yaml_path, split="test", imgsz=640, batch=16, workers=2)
